# 8.3. Preprocessing data

Taken from [scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-data)

## 8.3.1. Standardization, or mean removal and variance scaling
The preprocessing module provides the StandardScaler utility class, which is a quick and easy way to perform the following operation on an array-like dataset:

In [1]:
from sklearn import preprocessing
import numpy as np
X_train = np.array([[ 1., -1.,  2.],
                    [ 2.,  0.,  0.],
                    [ 0.,  1., -1.]])
scaler = preprocessing.StandardScaler().fit(X_train)
display(scaler)

display(scaler.mean_)

display(scaler.scale_)

X_scaled = scaler.transform(X_train)
display(X_scaled)

StandardScaler()

array([1.        , 0.        , 0.33333333])

array([0.81649658, 0.81649658, 1.24721913])

array([[ 0.        , -1.22474487,  1.33630621],
       [ 1.22474487,  0.        , -0.26726124],
       [-1.22474487,  1.22474487, -1.06904497]])

Scaled data has zero mean and unit variance:

In [2]:
display(X_scaled.mean(axis=0))

display(X_scaled.std(axis=0))

array([0., 0., 0.])

array([1., 1., 1.])

StandardScaler works column by column.

- Columns represent features, such as temperature or cost.
- Rows represent observations, such as months.

Each column uses its own mean and standard deviation for scaling.

### Mean and variance after standard scaling

`StandardScaler` scales each column independently using:

$$
z = \frac{x - \mu}{\sigma}
$$

where:
- $x$ is the original value.
- $\mu$ is the original column mean.
- $\sigma$ is the original column standard deviation.

### Example: first column

The original values are:

$$
[1,\ 2,\ 0]
$$

**1. Calculate the mean:**

$$
\mu = \frac{1 + 2 + 0}{3} = 1
$$

**2. Calculate the variance:**

Variance is the average squared distance from the mean:

$$
\sigma^2 = \frac{(1-1)^2 + (2-1)^2 + (0-1)^2}{3}
= \frac{2}{3}
$$

`StandardScaler` divides by the number of observations ($n$), rather than $n-1$.

**3. Calculate the standard deviation:**

$$
\sigma = \sqrt{\frac{2}{3}} \approx 0.8165
$$

**4. Scale each value:**

$$
\left[
\frac{1-1}{0.8165},\
\frac{2-1}{0.8165},\
\frac{0-1}{0.8165}
\right]
\approx [0,\ 1.2247,\ -1.2247]
$$

### Verify the scaled mean

$$
\mu_z = \frac{0 + 1.2247 - 1.2247}{3} = 0
$$

### Verify the scaled variance

Using the unrounded scaled values:

$$
\sigma_z^2
= \frac{(0-0)^2 + (\sqrt{3/2}-0)^2 + (-\sqrt{3/2}-0)^2}{3}
= \frac{0 + 1.5 + 1.5}{3}
= 1
$$

Subtracting the mean centres the column at zero. Dividing by the standard deviation gives it unit variance.

These properties apply to each nonconstant column of the data used to fit the scaler. Rounded displayed values may give results slightly different from 0 and 1.

This class implements the Transformer API to compute the mean and standard deviation on a training set so as to be able to later re-apply the same transformation on the testing set. This class is hence suitable for use in the early steps of a Pipeline:

In [3]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
pipe = make_pipeline(StandardScaler(), LogisticRegression())
pipe.fit(X_train, y_train)  # apply scaling on training data

display(pipe.score(X_test, y_test))  # apply scaling on testing data, without leaking training data.

0.96

## 8.3.1.1. Scaling features to a range

An alternative standardization is scaling features to lie between a given minimum and maximum value, often between zero and one, or so that the maximum absolute value of each feature is scaled to unit size. This can be achieved using MinMaxScaler or MaxAbsScaler, respectively.

The motivation to use this scaling includes robustness to very small standard deviations of features and preserving zero entries in sparse data.

Here is an example to scale a toy data matrix to the [0, 1] range:

In [4]:
X_train = np.array([[ 1., -1.,  2.],
                    [ 2.,  0.,  0.],
                    [ 0.,  1., -1.]])
min_max_scaler = preprocessing.MinMaxScaler()
X_train_minmax = min_max_scaler.fit_transform(X_train)
display(X_train_minmax)

array([[0.5       , 0.        , 1.        ],
       [1.        , 0.5       , 0.33333333],
       [0.        , 1.        , 0.        ]])

The same instance of the transformer can then be applied to some new test data unseen during the fit call: the same scaling and shifting operations will be applied to be consistent with the transformation performed on the train data:

In [5]:
X_test = np.array([[-3., -1.,  4.]])
X_test_minmax = min_max_scaler.transform(X_test)
display(X_test_minmax)

array([[-1.5       ,  0.        ,  1.66666667]])

It is possible to introspect the scaler attributes to find about the exact nature of the transformation learned on the training data:

In [6]:
display(min_max_scaler.scale_)

display(min_max_scaler.min_)

array([0.5       , 0.5       , 0.33333333])

array([0.        , 0.5       , 0.33333333])

If MinMaxScaler is given an explicit `feature_range=(min, max)` the full formula is:

In [7]:
range_min = 0
range_max = 1

X_std = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
X_scaled = X_std * (range_max - range_min) + range_min

display(X_scaled)

array([[0.12381866, 0.52439337, 0.4864274 , ..., 0.49285474, 0.67906612,
        0.21207425],
       [0.84375859, 0.64635665, 0.5435557 , ..., 0.82812533, 0.64916728,
        0.42922402],
       [0.55742129, 0.70255081, 0.81972143, ..., 0.79152641, 0.42087826,
        0.34748043],
       ...,
       [0.51666315, 0.20845695, 0.60900112, ..., 0.49782028, 0.43780975,
        0.39628525],
       [0.73933131, 0.7513872 , 0.46276529, ..., 0.49859413, 0.28347444,
        0.24459206],
       [0.52328794, 0.77176729, 0.53917657, ..., 0.75291143, 0.58023005,
        0.34009484]], shape=(100, 20))

MaxAbsScaler works in a very similar fashion, but scales in a way that the training data lies within the range [-1, 1] by dividing through the largest maximum value in each feature. It is meant for data that is already centered at zero or sparse data.

Here is how to use the toy data from the previous example with this scaler:

In [8]:
X_train = np.array([[ 1., -1.,  2.],
                    [ 2.,  0.,  0.],
                    [ 0.,  1., -1.]])
max_abs_scaler = preprocessing.MaxAbsScaler()
X_train_maxabs = max_abs_scaler.fit_transform(X_train)
display(X_train_maxabs)
X_test = np.array([[ -3., -1.,  4.]])
X_test_maxabs = max_abs_scaler.transform(X_test)
display(X_test_maxabs)
display(max_abs_scaler.scale_)

array([[ 0.5, -1. ,  1. ],
       [ 1. ,  0. ,  0. ],
       [ 0. ,  1. , -0.5]])

array([[-1.5, -1. ,  2. ]])

array([2., 1., 2.])

## 8.3.2.1. Mapping to a Uniform distribution

`QuantileTransformer provides a non-parametric transformation to map the data to a uniform distribution with values between 0 and 1:


In [9]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
quantile_transformer = preprocessing.QuantileTransformer(random_state=0,n_quantiles=112)
X_train_trans = quantile_transformer.fit_transform(X_train)
X_test_trans = quantile_transformer.transform(X_test)
#display(X_train_trans)
#display(X_test_trans)
display(np.percentile(X_train[:, 0], [0, 25, 50, 75, 100]))

array([4.3, 5.1, 5.8, 6.5, 7.9])

This feature corresponds to the sepal length in cm. Once the quantile transformation is applied, those landmarks approach closely the percentiles previously defined:


In [10]:
np.percentile(X_train_trans[:, 0], [0, 25, 50, 75, 100]).round(2)

array([0.  , 0.24, 0.51, 0.74, 1.  ])

This can be confirmed on an independent testing set with similar remarks:

In [11]:
display(np.percentile(X_test[:, 0], [0, 25, 50, 75, 100]).round(3))
display(np.percentile(X_test_trans[:, 0], [0, 25, 50, 75, 100]).round(2))

array([4.4  , 5.125, 5.75 , 6.175, 7.3  ])

array([0.01, 0.25, 0.48, 0.6 , 0.94])

## 8.3.2.2. Mapping to a Gaussian distribution


In [12]:
quantile_transformer = preprocessing.QuantileTransformer(
    output_distribution='normal', random_state=0)
X_trans = quantile_transformer.fit_transform(X)
display(quantile_transformer.quantiles_)

/Users/alvaro/Library/Caches/JetBrains/PyCharm2024.2/demo/PyCharmLearningProject/venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2912: UserWarning: n_quantiles (1000) is greater than the total number of samples (150). n_quantiles is set to n_samples.
  warnings.warn(


array([[4.3, 2. , 1. , 0.1],
       [4.4, 2.2, 1.1, 0.1],
       [4.4, 2.2, 1.2, 0.1],
       [4.4, 2.2, 1.2, 0.1],
       [4.5, 2.3, 1.3, 0.1],
       [4.6, 2.3, 1.3, 0.2],
       [4.6, 2.3, 1.3, 0.2],
       [4.6, 2.3, 1.3, 0.2],
       [4.6, 2.4, 1.3, 0.2],
       [4.7, 2.4, 1.3, 0.2],
       [4.7, 2.4, 1.3, 0.2],
       [4.8, 2.5, 1.4, 0.2],
       [4.8, 2.5, 1.4, 0.2],
       [4.8, 2.5, 1.4, 0.2],
       [4.8, 2.5, 1.4, 0.2],
       [4.8, 2.5, 1.4, 0.2],
       [4.9, 2.5, 1.4, 0.2],
       [4.9, 2.5, 1.4, 0.2],
       [4.9, 2.5, 1.4, 0.2],
       [4.9, 2.6, 1.4, 0.2],
       [4.9, 2.6, 1.4, 0.2],
       [4.9, 2.6, 1.4, 0.2],
       [5. , 2.6, 1.4, 0.2],
       [5. , 2.6, 1.4, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5. , 2.7, 1.5, 0.2],
       [5.1, 2.7, 1.5, 0.2],
       [5.1, 2.8, 1.5, 0.2],
       [5.1, 2

## 8.3.3. Normalization

Normalization is the process of scaling individual samples to have unit norm. This process can be useful if you plan to use a quadratic form such as the dot-product or any other kernel to quantify the similarity of any pair of samples.

This assumption is the base of the Vector Space Model often used in text classification and clustering contexts.

The function normalize provides a quick and easy way to perform this operation on a single array-like dataset, either using the l1, l2, or max norms:

In [13]:
X = [[ 1., -1.,  2.],
     [ 2.,  0.,  0.],
     [ 0.,  1., -1.]]
X_normalized = preprocessing.normalize(X, norm='l2')

display(X_normalized)

array([[ 0.40824829, -0.40824829,  0.81649658],
       [ 1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.70710678, -0.70710678]])

In [14]:
normalizer = preprocessing.Normalizer().fit(X)  # fit does nothing
display(normalizer)

Normalizer()

In [15]:
display(normalizer.transform(X))

display(normalizer.transform([[-1.,  1., 0.]]))

array([[ 0.40824829, -0.40824829,  0.81649658],
       [ 1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.70710678, -0.70710678]])

array([[-0.70710678,  0.70710678,  0.        ]])

## 8.3.4. Encoding categorical features

Often features are not given as continuous values but categorical. For example a person could have features ["male", "female"], ["from Europe", "from US", "from Asia"], ["uses Firefox", "uses Chrome", "uses Safari", "uses Internet Explorer"]. Such features can be efficiently coded as integers, for instance ["male", "from US", "uses Internet Explorer"] could be expressed as [0, 1, 3] while ["female", "from Asia", "uses Chrome"] would be [1, 2, 1].

To convert categorical features to such integer codes, we can use the OrdinalEncoder. This estimator transforms each categorical feature to one new feature of integers (0 to n_categories - 1):

In [16]:
enc = preprocessing.OrdinalEncoder()
X = [['male', 'from US', 'uses Safari'], ['female', 'from Europe', 'uses Firefox']]
display(enc.fit(X))
display(enc.transform([['female', 'from US', 'uses Safari']]))

OrdinalEncoder()

array([[0., 1., 1.]])

Such integer representation can, however, not be used directly with all scikit-learn estimators, as these expect continuous input, and would interpret the categories as being ordered, which is often not desired (i.e. the set of browsers was ordered arbitrarily).

By default, OrdinalEncoder will also passthrough missing values that are indicated by np.nan.

In [17]:
enc = preprocessing.OrdinalEncoder()
X = [['male'], ['female'], [np.nan], ['female']]
enc.fit_transform(X)

array([[ 1.],
       [ 0.],
       [nan],
       [ 0.]])

If you want codes assigned in your original order, specify it explicitly:

In [18]:
from sklearn import preprocessing

X = [
    ['male', 'from US', 'uses Safari'],
    ['female', 'from Europe', 'uses Firefox']
]

enc = preprocessing.OrdinalEncoder(categories=[
    ['male', 'female'],             # Column 1
    ["from Europe", "from US", "from Asia"],      # Column 2
    ["uses Firefox", "uses Chrome", "uses Safari", "uses Internet Explorer"]  # Column 3
])

enc.fit(X)

display(enc.transform([
    ['female', 'from US', 'uses Safari']
]))
# array([[1., 0., 0.]])

array([[1., 1., 2.]])

OrdinalEncoder provides a parameter encoded_missing_value to encode the missing values without the need to create a pipeline and using SimpleImputer.


In [19]:
enc = preprocessing.OrdinalEncoder(encoded_missing_value=-1)
X = [['male'], ['female'], [np.nan], ['female']]
enc.fit_transform(X)

array([[ 1.],
       [ 0.],
       [-1.],
       [ 0.]])

The above processing is equivalent to the following pipeline:



In [20]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
enc = Pipeline(steps=[
    ("encoder", preprocessing.OrdinalEncoder()),
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
])
enc.fit_transform(X)

array([[ 1.],
       [ 0.],
       [-1.],
       [ 0.]])

Another possibility to convert categorical features to features that can be used with scikit-learn estimators is to use a one-of-K, also known as one-hot or dummy encoding. This type of encoding can be obtained with the OneHotEncoder, which transforms each categorical feature with n_categories possible values into n_categories binary features, with one of them 1, and all others 0.

Continuing the example above:

In [21]:
enc = preprocessing.OneHotEncoder()
X = [['male', 'from US', 'uses Safari'], ['female', 'from Europe', 'uses Firefox']]
enc.fit(X)
enc.transform([['female', 'from US', 'uses Safari'],
               ['male', 'from Europe', 'uses Safari']]).toarray()

array([[1., 0., 0., 1., 0., 1.],
       [0., 1., 1., 0., 0., 1.]])

In [22]:
enc.categories_


[array(['female', 'male'], dtype=object),
 array(['from Europe', 'from US'], dtype=object),
 array(['uses Firefox', 'uses Safari'], dtype=object)]

In [23]:
genders = ['female', 'male']
locations = ['from Africa', 'from Asia', 'from Europe', 'from US']
browsers = ['uses Chrome', 'uses Firefox', 'uses IE', 'uses Safari']
enc = preprocessing.OneHotEncoder(categories=[genders, locations, browsers])
# Note that for there are missing categorical values for the 2nd and 3rd
# feature
X = [['male', 'from US', 'uses Safari'], ['female', 'from Europe', 'uses Firefox']]
enc.fit(X)
enc.transform([['female', 'from Asia', 'uses Chrome']]).toarray()

array([[1., 0., 0., 1., 0., 0., 1., 0., 0., 0.]])

In [24]:
enc = preprocessing.OneHotEncoder(handle_unknown='infrequent_if_exist')
X = [['male', 'from US', 'uses Safari'], ['female', 'from Europe', 'uses Firefox']]
enc.fit(X)
enc.transform([['female', 'from Asia', 'uses Chrome']]).toarray()

array([[1., 0., 0., 0., 0., 0.]])

It is also possible to encode each column into `n_categories - 1` columns instead of `n_categories` columns by using the drop parameter. This parameter allows the user to specify a category for each feature to be dropped. This is useful to avoid co-linearity in the input matrix in some classifiers. Such functionality is useful, for example, when using non-regularized regression (LinearRegression), since co-linearity would cause the covariance matrix to be non-invertible:


In [25]:
X = [['male', 'from US', 'uses Safari'],
     ['female', 'from Europe', 'uses Firefox']]
drop_enc = preprocessing.OneHotEncoder(drop='first').fit(X)
display(drop_enc.categories_)
display(drop_enc.transform(X).toarray())

[array(['female', 'male'], dtype=object),
 array(['from Europe', 'from US'], dtype=object),
 array(['uses Firefox', 'uses Safari'], dtype=object)]

array([[1., 1., 1.],
       [0., 0., 0.]])

In [26]:
X = [['male', 'US', 'Safari'],
     ['female', 'Europe', 'Firefox'],
     ['female', 'Asia', 'Chrome']]
drop_enc = preprocessing.OneHotEncoder(drop='if_binary').fit(X)
display(drop_enc.categories_)
display(drop_enc.transform(X).toarray())

[array(['female', 'male'], dtype=object),
 array(['Asia', 'Europe', 'US'], dtype=object),
 array(['Chrome', 'Firefox', 'Safari'], dtype=object)]

array([[1., 0., 0., 1., 0., 0., 1.],
       [0., 0., 1., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0., 0.]])

In [27]:
import pandas as pd

display(pd.DataFrame(
    drop_enc.transform(X).toarray(),
    columns=drop_enc.get_feature_names_out(
        ['gender', 'location', 'browser']
    )
))

,gender_male,location_Asia,location_Europe,location_US,browser_Chrome,browser_Firefox,browser_Safari
0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0


## Understanding the One-Hot Encoded Array

Original data:

| Row | Gender | Location | Browser |
|---|---|---|---|
| 1 | male | US | Safari |
| 2 | female | Europe | Firefox |
| 3 | female | Asia | Chrome |

The encoder is fitted using:

    drop_enc = preprocessing.OneHotEncoder(drop='if_binary').fit(X)

### What each column represents

`OneHotEncoder` creates indicator columns:

- **1** means the observation belongs to that category.
- **0** means it does not.

Categories are sorted alphabetically within each original feature. The output columns are:

| Column position | Category |
|---|---|
| 1 | male |
| 2 | Asia |
| 3 | Europe |
| 4 | US |
| 5 | Chrome |
| 6 | Firefox |
| 7 | Safari |

### Why there is no female column

`drop='if_binary'` removes the first category from features with exactly two categories.

Gender has two categories: `female` and `male`. The encoder drops `female`, leaving only the `male` column:

- `male = 1` represents male.
- `male = 0` represents female in this dataset.

Location and browser each have three categories, so all their columns are retained.

### Reading the output

    array([[1., 0., 0., 1., 0., 0., 1.],
           [0., 0., 1., 0., 0., 1., 0.],
           [0., 1., 0., 0., 1., 0., 0.]])

With column labels:

| Row | male | Asia | Europe | US | Chrome | Firefox | Safari |
|---|---|---|---|---|---|---|---|
| 1 | 1 | 0 | 0 | 1 | 0 | 0 | 1 |
| 2 | 0 | 0 | 1 | 0 | 0 | 1 | 0 |
| 3 | 0 | 1 | 0 | 0 | 1 | 0 | 0 |

Therefore:

- **Row 1:** male, from US, uses Safari.
- **Row 2:** female, from Europe, uses Firefox.
- **Row 3:** female, from Asia, uses Chrome.

### One-hot encoding versus ordinal encoding

Ordinal encoding assigns a category number. For location:

- Asia → 0
- Europe → 1
- US → 2

One-hot encoding creates a separate column for each category:

- Asia → `[1, 0, 0]`
- Europe → `[0, 1, 0]`
- US → `[0, 0, 1]`

US has category index 2, so one-hot encoding places a **1 at index 2** (the third position). It does not use 2 as the encoded value.

In the transformed X, the first column is the encoding of the feature with categories “male”/”female”, while the remaining 6 columns are the encoding of the 2 features with respectively 3 categories each.

When handle_unknown='ignore' and drop is not None, unknown categories will be encoded as all zeros:

In [28]:
drop_enc = preprocessing.OneHotEncoder(drop='first',
                                       handle_unknown='ignore').fit(X)
X_test = [['unknown', 'America', 'IE']]
drop_enc.transform(X_test).toarray()

/Users/alvaro/Library/Caches/JetBrains/PyCharm2024.2/demo/PyCharmLearningProject/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


array([[0., 0., 0., 0., 0.]])

In [29]:
drop_enc = preprocessing.OneHotEncoder(drop='if_binary', sparse_output=False,
                                       handle_unknown='ignore').fit(X)
X_test = [['unknown', 'America', 'IE']]
X_trans = drop_enc.transform(X_test)
display(X_trans)
display(drop_enc.inverse_transform(X_trans))

/Users/alvaro/Library/Caches/JetBrains/PyCharm2024.2/demo/PyCharmLearningProject/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


array([[0., 0., 0., 0., 0., 0., 0.]])

array([['female', None, None]], dtype=object)

## 8.3.4.1. Infrequent categories
OneHotEncoder and OrdinalEncoder support aggregating infrequent categories into a single output for each feature. The parameters to enable the gathering of infrequent categories are min_frequency and max_categories.

1. min_frequency is either an integer greater or equal to 1, or a float in the interval (0.0, 1.0). If min_frequency is an integer, categories with a cardinality smaller than min_frequency will be considered infrequent. If min_frequency is a float, categories with a cardinality smaller than this fraction of the total number of samples will be considered infrequent. The default value is 1, which means every category is encoded separately.
2. max_categories is either None or any integer greater than 1. This parameter sets an upper limit to the number of output features for each input feature. max_categories includes the feature that combines infrequent categories.

In the following example with OrdinalEncoder, the categories 'dog' and 'snake' are considered infrequent:

In [32]:
X = np.array([['dog'] * 5 + ['cat'] * 20 + ['rabbit'] * 10 +
              ['snake'] * 3], dtype=object).T
display(X)
enc = preprocessing.OrdinalEncoder(min_frequency=6).fit(X)
display(enc.infrequent_categories_)
display(enc.transform(np.array([['dog'], ['cat'], ['rabbit'], ['snake']])))

array([['dog'],
       ['dog'],
       ['dog'],
       ['dog'],
       ['dog'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['cat'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['rabbit'],
       ['snake'],
       ['snake'],
       ['snake']], dtype=object)

[array(['dog', 'snake'], dtype=object)]

array([[2.],
       [0.],
       [1.],
       [2.]])

In [33]:
import pandas as pd

categories = enc.categories_[0]
infrequent = enc.infrequent_categories_[0]

display(pd.DataFrame({
    "Category": categories,
    "Encoded value": enc.transform(categories.reshape(-1, 1)).ravel(),
    "Explanation": [
        "Infrequent: fewer than 6 occurrences; shares code 2"
        if category in infrequent
        else "Frequent: has its own code"
        for category in categories
    ]
}))

,Category,Encoded value,Explanation
0,cat,0.0,Frequent: has its own code
1,dog,2.0,Infrequent: fewer than 6 occurrences; shares c...
2,rabbit,1.0,Frequent: has its own code
3,snake,2.0,Infrequent: fewer than 6 occurrences; shares c...
